# Network Anomaly Detection — Section 15 of HTB Module 292

This notebook covers **only** the first network-anomaly section: what a random forest is and how to load/inspect HTB's modified NSL-KDD connection records. Later sections can add preprocessing, training, and evaluation without pretending they have happened here. Run with your `AI` Python environment.

An unusual connection is an **anomaly**, not automatically an intrusion. The dataset's `attack` field gives known labels, but a standard classifier trained on only `normal` rows cannot learn attack classes. Supervised intrusion classification and normal-only anomaly detection are different designs; HTB's introductory wording blends them.

## Random-forest voting, read aloud

A forest contains many decision trees. Each tree trains on a bootstrap sample—rows drawn **with replacement**—and considers random candidate features at each split. For rows A, B, C, D, a four-draw sample might be A, C, C, D: repeated C is allowed and B is missing. For classification, the trees vote. In `ŷ = mode(T₁(x), …, Tᴮ(x))`, say **y hat equals the most common prediction of tree one at x through tree B at x**. `x` is one connection row, `Tᵦ(x)` (tree sub b of x) is tree b's prediction, `B` is the number of trees, and `ŷ` (y hat) is the final predicted class. Votes normal, attack, attack yield attack. This section does **not** fit that forest yet.

## Use the archive linked by HTB

HTB supplies `https://cdn.services-k8s.prod.aws.htb.systems/content/modules/292/KDD_dataset.zip`. Save it as `KDD_dataset.zip` in your Downloads folder, or set `KDD_DATASET_ZIP` to its full local path. The archive stays outside Git. HTB's sample extracts every member into the working directory; we instead read only the expected `KDD+.txt` member directly, so a ZIP cannot place unexpected files in the project. The downloaded archive was verified to contain that one member with a valid CRC.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import pandas as pd

# A full-path environment override keeps the notebook portable across machines.
archive_path = Path(os.environ.get(
    'KDD_DATASET_ZIP', Path.home() / 'Downloads' / 'KDD_dataset.zip')).expanduser()
if not archive_path.is_file():
    raise FileNotFoundError(
        f'Missing {archive_path}. Download KDD_dataset.zip from the HTB lesson link.')
print('Archive:', archive_path.name, '(', archive_path.stat().st_size, 'bytes )')

## Name and load the 43 fields

Each CSV row represents one network **connection record**. The first 41 fields are features, such as `duration`, `protocol_type`, `src_bytes` (source bytes), and `dst_bytes` (destination bytes). `attack` is the known traffic label; `level` is an additional difficulty/metadata value, **not** a second attack label. Giving pandas `names=columns` makes the otherwise headerless file readable. We do not one-hot encode the categorical columns or create a target here; that belongs to later sections.

In [ ]:
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack', 'level'
]
assert len(columns) == 43
with ZipFile(archive_path) as archive:
    if 'KDD+.txt' not in archive.namelist():
        raise ValueError('Expected KDD+.txt member is absent from the HTB ZIP.')
    with archive.open('KDD+.txt') as dataset_file:
        df = pd.read_csv(dataset_file, names=columns)
assert df.shape[1] == len(columns) and len(df) > 0
assert df['attack'].notna().all()
print('Connection records:', len(df), '| columns:', df.shape[1])
print(df.head(3).to_string(index=False))

## First quality and label checks

A label count tells us which traffic types appear most often; it does not yet measure model performance. Missing values and dtypes guide later preprocessing. We inspect small summaries rather than printing every row or converting a large table to an unnecessarily bigger representation. The exact counts belong to this downloaded archive version.

In [ ]:
print('Top attack labels:')
print(df['attack'].value_counts().head(10).to_string())
print('Missing cells:', int(df.isna().sum().sum()))
print('Categorical examples:', df[['protocol_type', 'service', 'flag']].dtypes.to_dict())
print('Level range:', df['level'].min(), 'to', df['level'].max())
assert 'normal' in set(df['attack'])
assert df['attack'].nunique() > 1
assert pd.api.types.is_numeric_dtype(df['level'])

Section 15 ends here: the HTB ZIP is validated, 43 fields are named, and its connection records are inspected. No random forest is trained and no result is called an anomaly score yet. Later sections must define the target, transform categorical fields, split data without leakage, and evaluate before making detection claims.

# Section 16 — Preprocessing and Splitting the Dataset

We now create known-label targets, turn selected categories into numeric indicator columns, and reserve separate core-training, validation, and test rows. This is **supervised classification preparation**; no anomaly score or forest is fit in this section. Integer class IDs name categories; they do not measure attack severity. We retain HTB's feature selection and variable names where useful, but correct three attack-name mismatches observed in the provided archive and prevent unknown attacks from silently becoming normal.

## Binary and five-class targets

`attack_flag` is 0 exactly when `attack == 'normal'`, otherwise 1. `attack_map` groups the *known* attack strings into DoS, probe, privilege escalation, and access, coded 1 through 4; normal is 0. The HTB page spells three names differently from this archive: `loadmdoule`/`loadmodule`, `http_tunnel`/`httptunnel`, and `xclock`/`xlock`. The archive has 153 rows with those three actual names. HTB's catch-all `else: return 0` would mislabel them as normal, so we use the observed strings and fail on any unknown name. A new, unlisted attack must be investigated, not declared harmless.

In [ ]:
dos_attacks = ['apache2', 'back', 'land', 'neptune', 'mailbomb', 'pod',
               'processtable', 'smurf', 'teardrop', 'udpstorm', 'worm']
probe_attacks = ['ipsweep', 'mscan', 'nmap', 'portsweep', 'saint', 'satan']
privilege_attacks = ['buffer_overflow', 'loadmodule', 'perl', 'ps',
                     'rootkit', 'sqlattack', 'xterm']
access_attacks = ['ftp_write', 'guess_passwd', 'httptunnel', 'imap',
                  'multihop', 'named', 'phf', 'sendmail', 'snmpgetattack',
                  'snmpguess', 'spy', 'warezclient', 'warezmaster',
                  'xlock', 'xsnoop']
groups = {1: dos_attacks, 2: probe_attacks,
          3: privilege_attacks, 4: access_attacks}
attack_to_family = {'normal': 0}
for family_id, attack_names in groups.items():
    for attack_name in attack_names:
        if attack_name in attack_to_family:
            raise ValueError(f'Duplicate attack mapping: {attack_name}')
        attack_to_family[attack_name] = family_id
unknown_attacks = set(df['attack']) - set(attack_to_family)
if unknown_attacks:
    raise ValueError(f'Unmapped attack labels: {sorted(unknown_attacks)}')
df['attack_flag'] = df['attack'].ne('normal').astype('uint8')
df['attack_map'] = df['attack'].map(attack_to_family).astype('uint8')
assert (df['attack_flag'].eq(0) == df['attack'].eq('normal')).all()
assert (df['attack_map'].eq(0) == df['attack'].eq('normal')).all()
print('Binary counts:', df['attack_flag'].value_counts().sort_index().to_dict())
print('Five-class counts:', df['attack_map'].value_counts().sort_index().to_dict())

## Reserve test, then validation rows

`train_test_split` first reserves 20% of all rows as a test set. It then assigns 30% of the remaining 80% to validation: `0.80 × 0.30 = 0.24` (say eighty percent times thirty percent equals twenty-four percent of all rows). This leaves 56% core training, 24% validation, 20% test. `stratify` approximately preserves the five known classes in each partition. We split **row indices before learning one-hot columns**, so category vocabulary comes only from core training. `train_X`/`train_y` will name the whole 80% development portion; `multi_train_X`/`multi_train_y` will name its 56% core fit portion.

In [ ]:
from sklearn.model_selection import train_test_split

development_index, test_index = train_test_split(
    df.index.to_numpy(), test_size=0.20, random_state=1337,
    stratify=df['attack_map'])
core_index, validation_index = train_test_split(
    development_index, test_size=0.30, random_state=1337,
    stratify=df.loc[development_index, 'attack_map'])
assert len(core_index) + len(validation_index) + len(test_index) == len(df)
assert not (set(core_index) & set(validation_index))
assert not (set(core_index) & set(test_index))
assert not (set(validation_index) & set(test_index))
print('Core training:', len(core_index), 'validation:', len(validation_index),
      'test:', len(test_index))

## Encode two categorical fields and keep HTB's selected numeric fields

One-hot indicators express category membership: a TCP row has `protocol_type_tcp=1` and the other protocol indicators 0. Unlike assigning TCP=1 and UDP=2, this does **not** suggest that UDP is twice TCP. We learn column names from core training only and align all later rows to that schema. An unseen service at validation/test time gets zeros in the known service columns. HTB uses `protocol_type` and `service` plus the numeric list below; its other categorical `flag` and some other raw fields are omitted intentionally here. `level` is metadata and never a model feature or class label in this section.

In [ ]:
features_to_encode = ['protocol_type', 'service']
numeric_features = [
    'duration', 'src_bytes', 'dst_bytes', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells', 'num_access_files',
    'num_outbound_cmds', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate'
]
# Only the core training rows establish which dummy columns exist.
encoded = pd.get_dummies(
    df.loc[core_index, features_to_encode], dtype='uint8')
training_dummy_columns = encoded.columns

def feature_rows(row_index):
    indicators = pd.get_dummies(
        df.loc[row_index, features_to_encode], dtype='uint8')
    indicators = indicators.reindex(
        columns=training_dummy_columns, fill_value=0)
    # Joining by index keeps each row's network statistics paired with its indicators.
    return indicators.join(df.loc[row_index, numeric_features])

train_X = feature_rows(development_index)
test_X = feature_rows(test_index)
multi_train_X = train_X.loc[core_index]
multi_val_X = train_X.loc[validation_index]
multi_y = df['attack_map']
train_y = multi_y.loc[development_index]
test_y = multi_y.loc[test_index]
multi_train_y = multi_y.loc[core_index]
multi_val_y = multi_y.loc[validation_index]
assert train_X.columns.equals(test_X.columns)
assert multi_train_X.columns.equals(multi_val_X.columns)
assert train_X.index.equals(train_y.index)
assert test_X.index.equals(test_y.index)
assert multi_train_X.index.equals(multi_train_y.index)
assert multi_val_X.index.equals(multi_val_y.index)
assert 'attack' not in train_X and 'attack_map' not in train_X
assert 'level' not in train_X
print('Feature columns:', train_X.shape[1])
print('Development:', train_X.shape, 'test:', test_X.shape)
print('Core:', multi_train_X.shape, 'validation:', multi_val_X.shape)

Section 16 ends with aligned features and five-class labels for core training, validation, and test. The test rows have not fit the one-hot schema, and no forest has been trained. The random row split is a classroom evaluation setup; it does not establish performance against new time periods or attack families. Keep the test partition untouched until the later evaluation section.

# Section 17 — Training and Evaluation (Network Anomaly Detection)

We now fit the **supervised** five-class random forest HTB introduces, inspect validation performance, and only then evaluate on the reserved test rows. The model learns known labels: normal, DoS, probe, privilege escalation, and access. It does **not** become a normal-only novelty detector merely because the lesson uses the word anomaly. One worker (`n_jobs=1`) limits simultaneous tree-building memory; the default number of trees remains 100.

## How to read multiclass metrics

For class `k`, precision is `TPₖ/(TPₖ+FPₖ)`—say **true positives for class k divided by all predictions of k**. Recall is `TPₖ/(TPₖ+FNₖ)`—**correctly detected k rows divided by all actual k rows**. F1 is `2PR/(P+R)`—**two times precision times recall divided by their sum**. For P=1.0 and R=0.5, F1 is `1/1.5≈0.667`, so perfect precision cannot conceal low recall. `k` (sub k) means calculate separately for each category.

HTB prints weighted averages: `Σₖ (nₖ/N)F1ₖ`—say **sum over classes k of class k's share of all rows times its F-one**. `Σ` (capital sigma) means add; `nₖ` is class k's actual row count and `N` the total. With 99 common rows at F1=.99 and one rare row at F1=0, weighted F1=.9801. The privilege class is rare, so also inspect its own recall and the macro (equal-class-weight) average. A confusion-matrix cell `Cᵢⱼ` (C sub i j) counts actual class i predicted as class j; diagonal cells are correct.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model_multi = RandomForestClassifier(random_state=1337, n_jobs=1)
# Formula connection: each tree learns split questions from bootstrapped rows;
# the forest combines its trees' class votes for each 106-column input row.
rf_model_multi.fit(multi_train_X, multi_train_y)
assert rf_model_multi.n_estimators == 100
assert set(rf_model_multi.classes_) == {0, 1, 2, 3, 4}
assert len(rf_model_multi.feature_names_in_) == multi_train_X.shape[1]
print('Fitted trees:', len(rf_model_multi.estimators_))

## Validation: inspect mistakes before touching the test set

The rows of the five-by-five matrix are actual labels; columns are predictions. Diagonal counts are right, off-diagonal counts show which families were confused. HTB plots a heatmap; we print a labeled matrix to keep the notebook nonblocking and lighter on memory. `support` in the classification report is the number of actual examples of each class. HTB emphasizes weighted precision/recall/F1; we also show macro F1 because it gives the rare privilege class equal influence. `zero_division=0` makes a class with no predicted positives visible as zero precision.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

class_ids = [0, 1, 2, 3, 4]
class_labels = ['Normal', 'DoS', 'Probe', 'Privilege', 'Access']
multi_predictions = rf_model_multi.predict(multi_val_X)
accuracy = accuracy_score(multi_val_y, multi_predictions)
precision = precision_score(multi_val_y, multi_predictions,
                            labels=class_ids, average='weighted', zero_division=0)
recall = recall_score(multi_val_y, multi_predictions,
                      labels=class_ids, average='weighted', zero_division=0)
f1 = f1_score(multi_val_y, multi_predictions,
              labels=class_ids, average='weighted', zero_division=0)
macro_f1 = f1_score(multi_val_y, multi_predictions,
                    labels=class_ids, average='macro', zero_division=0)
conf_matrix = confusion_matrix(multi_val_y, multi_predictions, labels=class_ids)
print(f'Validation accuracy={accuracy:.4f}, weighted P={precision:.4f}, '
      f'R={recall:.4f}, F1={f1:.4f}, macro F1={macro_f1:.4f}')
print('Validation confusion matrix (actual rows, predicted columns):')
print(pd.DataFrame(conf_matrix, index=class_labels, columns=class_labels).to_string())
print(classification_report(multi_val_y, multi_predictions, labels=class_ids,
                            target_names=class_labels, zero_division=0))
assert len(multi_predictions) == len(multi_val_y)
assert int(conf_matrix.sum()) == len(multi_val_y)
assert all(0 <= metric <= 1 for metric in (accuracy, precision, recall, f1, macro_f1))

## One final report on the reserved test rows

Nothing in the model or preprocessing is changed after the validation report. The test scores describe this one random-row split, not guaranteed performance on a future network or novel attack. We again print the per-class report so a high weighted score cannot hide rare-class misses. Do not tune the forest after seeing this test result; doing so would turn the test partition into another validation set.

In [ ]:
test_multi_predictions = rf_model_multi.predict(test_X)
test_accuracy = accuracy_score(test_y, test_multi_predictions)
test_precision = precision_score(test_y, test_multi_predictions,
                                 labels=class_ids, average='weighted', zero_division=0)
test_recall = recall_score(test_y, test_multi_predictions,
                           labels=class_ids, average='weighted', zero_division=0)
test_f1 = f1_score(test_y, test_multi_predictions,
                   labels=class_ids, average='weighted', zero_division=0)
test_macro_f1 = f1_score(test_y, test_multi_predictions,
                         labels=class_ids, average='macro', zero_division=0)
test_conf_matrix = confusion_matrix(test_y, test_multi_predictions,
                                    labels=class_ids)
print(f'Test accuracy={test_accuracy:.4f}, weighted P={test_precision:.4f}, '
      f'R={test_recall:.4f}, F1={test_f1:.4f}, macro F1={test_macro_f1:.4f}')
print('Test confusion matrix (actual rows, predicted columns):')
print(pd.DataFrame(test_conf_matrix, index=class_labels, columns=class_labels).to_string())
print(classification_report(test_y, test_multi_predictions, labels=class_ids,
                            target_names=class_labels, zero_division=0))
assert len(test_multi_predictions) == len(test_y)
assert int(test_conf_matrix.sum()) == len(test_y)
assert all(0 <= metric <= 1 for metric in
           (test_accuracy, test_precision, test_recall, test_f1, test_macro_f1))

## Demonstrate saving without committing a model binary

`joblib.dump` serializes the fitted forest. The file is created in a temporary directory under the current writable folder and removed automatically. This checks that the fitted object can be written without leaving a large `.joblib` artifact in Git. It does not load any untrusted file; Python model deserialization can execute code. A later HTB evaluator section may export and upload the fitted model to its active target.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import joblib

with TemporaryDirectory(prefix='htb_kdd_study_', dir=Path.cwd()) as temporary_dir:
    model_path = Path(temporary_dir) / 'network_anomaly_detection_model.joblib'
    joblib.dump(rf_model_multi, model_path)
    assert model_path.is_file() and model_path.stat().st_size > 0
    print('Serialized forest size:', model_path.stat().st_size, 'bytes')
print('Temporary model file removed')

Section 17 ends with a fitted five-class random forest, validation and test reports, and a temporary model-export demonstration. The model classifies **known labeled categories**; it is not proven to detect never-before-seen attacks. HTB's heatmap is represented here by labeled text matrices, and both weighted and per-class results must be read before making a security claim. No target address, flag, dataset copy, or model binary is committed.

# Section 18 — Model Evaluation (Network Anomaly Detection)

HTB's evaluator has hidden examples. We upload the **already fitted** `rf_model_multi` from Section 17; this section does not retrain or tune it after seeing the reserved test result. `joblib.dump` turns that Python object into a file, and `requests.post(..., files={'model': model_file})` sends it as multipart form data. The form key must be `model`. On Pwnbox the URL is `http://localhost:8001/api/upload`; from your own machine it is `http://<VM-IP>:8001/api/upload` after spawning the VM and connecting HTB VPN. Port 8001 is specific to this network-model evaluator.

**Boundary:** `RUN_HTB_MODEL_UPLOAD` is false by default. Replace the placeholder with the active HTB VM address and enable the switch only for this authorized lab target. A temporary file is removed after the request. A valid HTTP response means the server received a request, not necessarily that the model met its hidden threshold; inspect the returned JSON for its actual result. Never load a `.joblib` file from an untrusted source, because deserialization can execute code. We do not save a target address, response, flag, or model binary in the notebook.

In [ ]:
import json
import requests

RUN_HTB_MODEL_UPLOAD = False  # Set True only for the active, authorized HTB VM.
HTB_EVALUATOR_URL = 'http://<VM-IP>:8001/api/upload'

if RUN_HTB_MODEL_UPLOAD:
    if '<VM-IP>' in HTB_EVALUATOR_URL:
        raise ValueError('Replace <VM-IP> with the active HTB VM address first.')
    if not HTB_EVALUATOR_URL.endswith(':8001/api/upload'):
        raise ValueError('Check the HTB network-evaluator port and endpoint.')
    # Reuse the fitted forest; do not refit using validation or test rows.
    with TemporaryDirectory(prefix='htb_kdd_upload_', dir=Path.cwd()) as temporary_dir:
        model_path = Path(temporary_dir) / 'network_anomaly_detection_model.joblib'
        joblib.dump(rf_model_multi, model_path)
        # Multipart field name 'model' implements the evaluator's upload contract.
        with model_path.open('rb') as model_file:
            response = requests.post(HTB_EVALUATOR_URL, files={'model': model_file}, timeout=120)
        response.raise_for_status()
        print(json.dumps(response.json(), indent=2))
    # TemporaryDirectory removes the serialized file, even if the request fails.
else:
    print('HTB model upload disabled; local model and test results are unchanged.')

The local validation/test reports and HTB's hidden evaluation answer different questions. Local scores show behavior on our reserved random split; only the HTB server can judge its undisclosed challenge set. This cell is intentionally non-submitting by default. Section 18 adds no new model-fitting step.